# Sola Face LoRA — Colab (OOM-safe)

**Твоя помилка `exit -9`** = процес убило через брак **системної RAM** під час завантаження SDXL (не баг датасету).

Цей ноутбук:
1. Прибирає TensorFlow (жер RAM)
2. Качає SDXL як **один** `.safetensors`
3. Тренує з `--lowram`

**Бажано:** Runtime → Change runtime type → GPU **+ High-RAM** (якщо є).

Upload: `datasets/sola_face_kohya.zip`


In [ ]:
# @title 0) GPU + free RAM
!nvidia-smi
import torch, os, gc
assert torch.cuda.is_available(), 'Enable GPU'
print(torch.cuda.get_device_name(0))
# TensorFlow on Colab eats RAM and helps cause exit -9
!pip -q uninstall -y tensorflow tensorflow-cpu tensorflow-gpu keras tensorboard tb-nightly || true
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect()
!free -h


In [ ]:
# @title 1) Install Kohya (light)
import os, shutil
os.chdir('/content')
!pip -q install -U pip wheel
!pip -q install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
shutil.rmtree('/content/sd-scripts', ignore_errors=True)
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
os.chdir('/content/sd-scripts')
!pip -q install accelerate==0.31.0 transformers==4.41.2 diffusers==0.29.2 safetensors==0.4.3
!pip -q install ftfy einops opencv-python-headless toml voluptuous bitsandbytes==0.43.1 huggingface_hub
print('OK', os.path.isfile('sdxl_train_network.py'))


In [ ]:
# @title 2) Download SDXL as SINGLE safetensors (less RAM than Diffusers pipeline)
import os
from huggingface_hub import hf_hub_download
os.makedirs('/content/models', exist_ok=True)
CKPT = '/content/models/sd_xl_base_1.0_fp16.safetensors'
if not os.path.isfile(CKPT) or os.path.getsize(CKPT) < 1_000_000_000:
    # fp16 weights ~6GB
    path = hf_hub_download(
        repo_id='stabilityai/stable-diffusion-xl-base-1.0',
        filename='sd_xl_base_1.0_0.9vae.safetensors',
        local_dir='/content/models',
        local_dir_use_symlinks=False,
    )
    # normalize name
    import shutil
    if path != CKPT:
        shutil.move(path, CKPT) if not os.path.isfile(CKPT) else None
        if os.path.isfile(path) and path != CKPT:
            CKPT = path
print('CKPT', CKPT, 'GB', round(os.path.getsize(CKPT)/1e9,2))
!free -h


In [ ]:
# @title 3) Upload dataset zip + use 2 repeats (less cache pressure)
import os, zipfile, shutil
from google.colab import files
DATA='/content/sola_data'
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(DATA, exist_ok=True); os.chdir(DATA)
print('Upload sola_face_kohya.zip')
up=files.upload(); assert up
for name in up:
  if name.lower().endswith('.zip'):
    zipfile.ZipFile(name).extractall(DATA)
src=None
for r,ds,_ in os.walk(DATA):
  if '10_sola_face' in ds:
    src=os.path.join(r,'10_sola_face'); break
assert src, '10_sola_face not found'
# Kohya reads repeats from folder prefix: 2_sola_face => 2 repeats (was 10 → heavier)
dst=os.path.join(DATA,'2_sola_face')
if os.path.abspath(src)!=os.path.abspath(dst):
  shutil.rmtree(dst, ignore_errors=True)
  shutil.copytree(src, dst)
TRAIN_ROOT=DATA
OUT='/content/outputs/sola_face_lora'; os.makedirs(OUT, exist_ok=True)
n=len([f for f in os.listdir(dst) if f.lower().endswith('.jpg')])
print('TRAIN_ROOT', TRAIN_ROOT, 'class', dst, 'images', n)
assert n>=10
# discover ckpt path
import glob
cands=glob.glob('/content/models/*.safetensors')
CKPT=sorted(cands, key=os.path.getsize)[-1]
print('Using CKPT', CKPT, round(os.path.getsize(CKPT)/1e9,2), 'GB')


In [ ]:
# @title 4) Train (single-file CKPT + lowram) — LIVE LOG
import os, sys, subprocess, glob, gc
import torch
gc.collect(); torch.cuda.empty_cache()

os.chdir('/content/sd-scripts')
LOG=OUT+'/train.log'

cmd=[
  sys.executable,'sdxl_train_network.py',
  f'--pretrained_model_name_or_path={CKPT}',
  f'--train_data_dir={TRAIN_ROOT}',
  f'--output_dir={OUT}',
  '--output_name=sola_face_sdxl',
  '--save_model_as=safetensors',
  '--save_precision=fp16',
  '--caption_extension=.txt',
  '--resolution=512,512',
  '--enable_bucket',
  '--min_bucket_reso=256',
  '--max_bucket_reso=1024',
  '--train_batch_size=1',
  '--gradient_checkpointing',
  '--lowram',
  '--max_train_epochs=6',
  '--save_every_n_epochs=2',
  '--learning_rate=1e-4',
  '--unet_lr=1e-4',
  '--text_encoder_lr=5e-5',
  '--lr_scheduler=cosine',
  '--lr_warmup_steps=20',
  '--optimizer_type=AdamW8bit',
  '--network_module=networks.lora',
  '--network_dim=8',
  '--network_alpha=8',
  '--mixed_precision=fp16',
  '--cache_latents',
  '--cache_latents_to_disk',
  '--seed=42',
  '--keep_tokens=1',
  '--max_data_loader_n_workers=0',
  '--network_train_unet_only',
]

env=os.environ.copy()
env['PYTHONPATH']='/content/sd-scripts'+os.pathsep+env.get('PYTHONPATH','')
env['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
env['TF_CPP_MIN_LOG_LEVEL']='3'

print('CMD', ' '.join(cmd), flush=True)
!free -h
with open(LOG,'w',encoding='utf-8') as logf:
  logf.write('CMD '+' '.join(cmd)+'\n')
  p=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd='/content/sd-scripts')
  for line in p.stdout:
    print(line, end='')
    logf.write(line)
  code=p.wait()

paths=glob.glob(OUT+'/**/*.safetensors', recursive=True)
print('exit', code, 'safetensors', paths)
if code==-9 or code==137:
  raise RuntimeError('OOM kill (exit -9/137). Увімкни High-RAM runtime або Colab Pro GPU і перезапусти з нуля.')
if code!=0 or not paths:
  print(open(LOG,encoding='utf-8',errors='replace').read()[-8000:])
  raise RuntimeError('Train failed — paste log tail above')
print('SUCCESS')


In [ ]:
# @title 5) Download LoRA
import glob, os
from google.colab import files
paths=sorted(glob.glob('/content/outputs/sola_face_lora/**/*.safetensors', recursive=True), key=os.path.getmtime)
print(paths)
assert paths
best=[p for p in paths if 'sola_face_sdxl' in os.path.basename(p)] or paths
files.download(best[-1])
